In [1]:
from pydantic import BaseModel
class User(BaseModel):
    id:int
    name:str
    is_active:bool

input_data={'id':1,'name':'Nithya Sai','is_active':True}

user1=User(**input_data)


In [2]:
user1

User(id=1, name='Nithya Sai', is_active=True)

In [4]:
#nested model
class Address(BaseModel):
    street: str
    city: str
    country:str
class User(BaseModel):
    id:int
    name:str
    address:Address

address=Address(street="Gandhi street",city="Hyderabad",country="India")

user_data={
    "id":"123",
    "name":"raju",
    "address":address

}

user=User(**user_data)
print(user)
print(user.address
      )

id=123 name='raju' address=Address(street='Gandhi street', city='Hyderabad', country='India')
street='Gandhi street' city='Hyderabad' country='India'


In [5]:
class Product(BaseModel):
    id:int
    name:str
    price:float
    in_stock:bool

prod1=Product(id=1,name="Chair",price=29.09,in_stock=True)
prod2=Product(id=2,name="bed")

ValidationError: 2 validation errors for Product
price
  Field required [type=missing, input_value={'id': 2, 'name': 'bed'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
in_stock
  Field required [type=missing, input_value={'id': 2, 'name': 'bed'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
#self referencing
from typing import List,Optional
from pydantic import BaseModel

class Comment(BaseModel):
    id:int
    content:str
    replies: Optional[List['Comment']]=None

#in the above code we pass the Class name as string literal to avoid the run time error. 
# and below code will tell the interpreter to use the Class as the filler 
Comment.model_rebuild()

comment=Comment(
    id=1,
    content="Hey! how do you do?",
    replies=[
        Comment(id=2,content="I am good"),
        Comment(id=3,content="Hey dude",replies=[Comment(id=4,content="I am dude")])
    ]
)
comment

Comment(id=1, content='Hey! how do you do?', replies=[Comment(id=2, content='I am good', replies=None), Comment(id=3, content='Hey dude', replies=[Comment(id=4, content='I am dude', replies=None)])])

In [3]:
#Field validators and Model validator
from pydantic import BaseModel,field_validator,model_validator

class User(BaseModel):
    username:str

    @field_validator('username')
    def username_length(cls,v):
        if len(v)<4:
            raise ValueError("usernamemust be atleast 4 characters")
        return v
    
class SignupData(BaseModel):
    password:str
    confirm_password:str

    @model_validator(mode='after')
    def password_match(cls,values):
        if values.password !=values.confirm_password:
            raise ValueError("Password do not match")
        return values


/tmp/ipykernel_7470/1867482441.py:18: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  def password_match(cls,values):


In [5]:
user=User(username='opi')

ValidationError: 1 validation error for User
username
  Value error, usernamemust be atleast 4 characters [type=value_error, input_value='opi', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [6]:
passw_ord=SignupData(password='loiu',confirm_password='ksjj')

ValidationError: 1 validation error for SignupData
  Value error, Password do not match [type=value_error, input_value={'password': 'loiu', 'confirm_password': 'ksjj'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [9]:
#typing module
from typing import List,Dict,Optional

class Cart(BaseModel):
    user_id:int
    items:List[str]
    quantities: Dict[str,int]

class BlogPosts(BaseModel):
    title:str
    content:str
    image_url:Optional[str]=None

cart_data={
    'user_id':"123",
    'items':['Laptop','Chair','Table'],
    'quantities':{
        'laptop':20,'Chair':12,'Table':13
    }
}

cart=Cart(**cart_data)

In [10]:
cart

Cart(user_id=123, items=['Laptop', 'Chair', 'Table'], quantities={'laptop': 20, 'Chair': 12, 'Table': 13})

In [22]:
#working with Field 
#every attribute is a field,but we can use conditions also

from pydantic import Field
import re

class Employee(BaseModel):
    id:int
    name:str = Field(...,min_length=3,max_length=25,description="Employee Name",examples="Rajesh")
    department:Optional[str] ='general'
    salary:float = Field(...,ge=10000)

class User_Fields(BaseModel):
    email:str= Field(...,pattern=r'')
    phone:str =Field(...,pattern=r'')
    age:int = Field(...,ge=0,le=120,description="Age in Years")
    discount: float=Field(...,ge=0,le=100,description="Discount percentage")




In [14]:
emp=Employee(id=1,name="raj",salary=12092)

In [15]:
emp

Employee(id=1, name='raj', department='general', salary=12092.0)

In [ ]:
user=User_Fields(email='',phone=' ',age=10,discount=19)

ValidationError: 1 validation error for User_Fields
email
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [17]:
user

User_Fields(email=' ', phone=' ', age=10, discount=19.0)

In [23]:
#check regex pattern